# Эволюционные алгоритмы

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Эволюционные алгоритмы».

## Что делает метод

Эволюционные алгоритмы — методы оптимизации, имитирующие естественный отбор. Поддерживается популяция решений-кандидатов; в каждом поколении лучшие отбираются, скрещиваются и мутируют, порождая новое поколение. Со временем популяция сдвигается к хорошим решениям.

Главное достоинство — нетребовательность: целевая функция не обязана быть гладкой, дифференцируемой или заданной формулой. Достаточно уметь вычислять её значение. Это подходит для подбора параметров эксперимента, конфигураций установки, расписаний.

В этом блокноте мы реализуем эволюционный алгоритм на numpy и минимизируем функцию с множеством локальных минимумов. Расчётное время прохождения — около пяти минут.

## Интуиция за методом

Представьте задачу селекции: есть множество сортов растения с разными свойствами, и нужно вывести лучший. Агроном отбирает самые перспективные, скрещивает их, а в следующем поколении появляются потомки — с чертами обоих родителей плюс случайными небольшими мутациями. Поколение за поколением популяция улучшается.

Эволюционный алгоритм делает ровно то же самое, но с числовыми решениями:

1. **Популяция** — набор случайных кандидатов (точек в пространстве параметров).
2. **Приспособленность** — значение целевой функции: насколько хорошо данная точка решает задачу.
3. **Отбор** — лучшие кандидаты получают больший шанс стать «родителями».
4. **Скрещивание** — новый кандидат создаётся как смесь двух «родителей».
5. **Мутация** — к новому кандидату добавляется небольшое случайное возмущение. Без мутации популяция быстро застряла бы в одной точке.
6. **Элитизм** — лучшая особь каждого поколения переходит в следующее без изменений, гарантируя, что лучшее найденное решение не будет потеряно.

Главное преимущество: целевая функция не обязана быть гладкой или дифференцируемой — достаточно уметь вычислять её значение.

## 1. Установка библиотек

In [ ]:
%pip install -q numpy pandas matplotlib

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационная задача

**Функция Растригина** — стандартный трудный тест для оптимизаторов: глобальный минимум в точке (0, 0) со значением 0, но вокруг него — множество локальных минимумов, разделённых барьерами. Метод, просто идущий «вниз по склону» (градиентный спуск), с высокой вероятностью застрянет в одном из них.

На этой задаче хорошо виден главный аргумент в пользу эволюционного алгоритма: популяция «смотрит» на функцию сразу в нескольких точках, что позволяет выбраться из локальных ловушек.

Ячейка определяет функцию и параметры области поиска.

In [ ]:
import numpy as np

def rastrigin(X):
    """Функция Растригина: много локальных минимумов, глобальный в нуле."""
    X = np.atleast_2d(X)
    return (10 * X.shape[1]
            + np.sum(X ** 2 - 10 * np.cos(2 * np.pi * X), axis=1))

bounds = (-5.12, 5.12)      # область поиска по каждой координате
dim = 2
print(f"Минимизируем функцию Растригина в области "
      f"[{bounds[0]}, {bounds[1]}]^{dim}")
print(f"Известный глобальный минимум: значение 0 в точке (0, 0)")

## 4. Применение метода

### Шаг 1. Начальная популяция

Случайно размещаем `pop_size` точек в области поиска. Это нулевое поколение — никакой информации об оптимуме ещё нет. Для каждой точки вычисляем приспособленность (значение функции). Чем меньше значение, тем лучше кандидат.

Ячейка выводит лучшее значение среди случайных начальных кандидатов — ориентир для сравнения с результатом после эволюции.

In [ ]:
# Фиксируем зерно для воспроизводимости
rng = np.random.default_rng(42)

pop_size = 80
population = rng.uniform(bounds[0], bounds[1], size=(pop_size, dim))
scores = rastrigin(population)
print(f"Размер популяции: {pop_size}")
print(f"Лучшее значение в начальной популяции: {scores.min():.3f}")

### Шаг 2. Операторы эволюции

Три функции, реализующие биологические аналогии:

- **Турнирный отбор**: из нескольких случайных кандидатов в «родители» выбирается лучший по приспособленности. Параметр `k` — «размер турнира»: чем больше, тем сильнее «давление отбора» (лучшие почти всегда побеждают) и тем быстрее популяция теряет разнообразие.
- **Скрещивание**: координаты потомка — случайная взвешенная смесь координат двух родителей. Наследуются черты обоих.
- **Мутация**: к каждой координате потомка с вероятностью `rate` добавляется случайное возмущение из нормального распределения с масштабом `scale`. Без мутации эволюция прекращается, как только все кандидаты стали похожими.

Эта ячейка только определяет функции; эволюционный цикл — в следующей ячейке.

In [ ]:
def tournament(pop, scores, rng, k=3):
    """Турнирный отбор: лучший из k случайных кандидатов."""
    idx = rng.integers(len(pop), size=k)
    return pop[idx[np.argmin(scores[idx])]]


def crossover(parent_a, parent_b, rng):
    """Скрещивание: координаты потомка - смесь родителей."""
    w = rng.random(parent_a.shape)
    return w * parent_a + (1 - w) * parent_b


def mutate(child, rng, rate=0.3, scale=0.6):
    """Мутация: случайное возмущение координат."""
    mask = rng.random(child.shape) < rate
    child = child + mask * rng.normal(0, scale, child.shape)
    return np.clip(child, bounds[0], bounds[1])

### Шаг 3. Эволюционный цикл по поколениям

Запускаем 60 поколений. В каждом поколении:
1. Оцениваем приспособленность всей популяции.
2. Лучшая особь (элита) переходит в следующее поколение без изменений.
3. Остальные особи создаются через турнирный отбор, скрещивание и мутацию.

Записываем историю лучшего и среднего значения по каждому поколению — из неё видно, как идёт сходимость.

После запуска выведутся найденные координаты минимума и значение функции в нём. Сравните с известным ответом: точка (0, 0), значение 0.

In [ ]:
n_generations = 60
best_history, mean_history = [], []
best_point, best_score = None, np.inf

for gen in range(n_generations):
    scores = rastrigin(population)
    gen_best = scores.argmin()
    if scores[gen_best] < best_score:
        best_score = scores[gen_best]
        best_point = population[gen_best].copy()
    best_history.append(best_score)
    mean_history.append(scores.mean())

    # Элитизм: лучшая особь переходит в новое поколение без изменений.
    new_pop = [population[gen_best].copy()]
    while len(new_pop) < pop_size:
        pa = tournament(population, scores, rng)
        pb = tournament(population, scores, rng)
        child = mutate(crossover(pa, pb, rng), rng)
        new_pop.append(child)
    population = np.array(new_pop)

print(f"Найденный минимум: значение {best_score:.4f}")
print(f"в точке ({best_point[0]:.3f}, {best_point[1]:.3f})")

### Шаг 4. Визуализация: область поиска и сходимость

Строим два графика:
- **Левый**: ландшафт функции Растригина (цветовая карта) и расположение финальной популяции. Хороший знак — точки сосредоточены у центра.
- **Правый**: история лучшего и среднего значения по поколениям. Это кривая сходимости алгоритма.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.5))

# Линии уровня функции и найденный минимум.
g = np.linspace(bounds[0], bounds[1], 200)
G1, G2 = np.meshgrid(g, g)
Z = rastrigin(np.column_stack([G1.ravel(), G2.ravel()])).reshape(G1.shape)
cs = axes[0].contourf(G1, G2, Z, levels=25, cmap="viridis")
axes[0].scatter(population[:, 0], population[:, 1], color="white",
                edgecolors=VIZ["node_text"], s=30,
                label="Финальная популяция")
axes[0].scatter(*best_point, color=VIZ["series"][2], marker="*", s=320,
                edgecolors="white", label="Найденный минимум")
axes[0].set_title("Область поиска и популяция")
axes[0].set_xlabel("Координата 1")
axes[0].set_ylabel("Координата 2")
axes[0].legend(loc="upper right")
fig.colorbar(cs, ax=axes[0], label="Значение функции")

axes[1].plot(best_history, color=VIZ["series"][0], label="Лучшее в популяции")
axes[1].plot(mean_history, color=VIZ["series"][3], label="Среднее по популяции")
axes[1].set_title("Сходимость по поколениям")
axes[1].set_xlabel("Поколение")
axes[1].set_ylabel("Значение функции")
axes[1].legend(loc="upper right")

fig.tight_layout()
plt.show()

**Как читать эти графики.**

Левый — ландшафт задачи:
- Тёмные области — низкие значения функции (хорошие решения), светлые — высокие (плохие).
- Центр (0, 0) — глобальный минимум, вокруг него видны «волны» локальных минимумов.
- Белые точки — финальная популяция: если алгоритм сошёлся, большинство точек должны быть рядом с центром.
- Звезда — найденное лучшее решение.

Правый — сходимость:
- Синяя линия («лучшее») монотонно убывает: элитизм гарантирует, что лучший результат не ухудшается.
- Серая линия («среднее») убывает с колебаниями: мутации вносят шум, но общее направление — вниз.
- Когда обе линии сходятся и перестают убывать — популяция исчерпала разнообразие. Это сигнал: можно увеличить мутацию или размер популяции.

## 5. Интерпретация результата

- **Найденный минимум** должен быть близок к точке (0, 0) со значением около нуля. Эволюционный поиск преодолевает локальные минимумы за счёт популяции и мутаций, тогда как градиентный спуск из случайной точки чаще застревает.
- **Левый график**: финальная популяция концентрируется у глобального минимума — признак сходимости. Если точки рассеяны, поколений или давления отбора недостаточно.
- **Правый график**: «лучшее в популяции» монотонно убывает (элитизм гарантирует неухудшение); «среднее» убывает с колебаниями из-за мутаций. Сближение двух кривых означает, что разнообразие популяции исчерпано.
- **Баланс параметров**: сильная мутация поддерживает разнообразие, но мешает точной настройке; слабая ускоряет сходимость, но повышает риск застрять. `rate`, `scale`, размер популяции и турнира подбирают под задачу.

Помните: эволюционный алгоритм стохастичен и не гарантирует глобальный оптимум. Несколько запусков с разными `seed` повышают надёжность.

## 6. Поэкспериментируйте сами

Измените один параметр, повторно запустите ячейки раздела 4 и сравните результат.

**Что менять и что наблюдать:**
- Уменьшите `pop_size` с 80 до 10. Будет ли алгоритм столь же надёжно находить глобальный минимум? Запустите несколько раз с разными `seed`.
- Установите `scale=0.05` в функции `mutate`. Популяция будет искать точнее, но медленнее — следите за правым графиком сходимости.
- Установите `scale=3.0`. Мутации станут настолько сильными, что потомки будут «прыгать» по всей области — хаос вместо поиска.
- Увеличьте `n_generations` до 200. Продолжает ли алгоритм улучшаться после 60 поколений или достиг плато?
- Измените размер турнира `k=3` на `k=10` внутри `tournament`. Как усиление давления отбора влияет на скорость сходимости и риск застрять в локальном минимуме?

In [ ]:
# --- Шаблон собственной целевой функции ---
# def my_objective(X):
#     X = np.atleast_2d(X)
#     # верните массив значений (по строке на кандидата); меньше - лучше
#     return ...
#
# dim = 3
# bounds = (0.0, 1.0)
# Затем в ячейках раздела 4 замените rastrigin на my_objective.

## Краткие выводы

- Эволюционный алгоритм работает с популяцией кандидатов и улучшает её через отбор, скрещивание и мутацию — имитируя естественный отбор.
- Ключевые операторы: отбор (кто становится родителем), скрещивание (как комбинируются решения) и мутация (что вносит разнообразие).
- Элитизм гарантирует, что лучшее найденное решение не ухудшается от поколения к поколению.
- Метод справляется с задачами, где целевая функция негладкая, имеет много локальных минимумов или задана как «чёрный ящик».
- Алгоритм стохастичен: разные запуски могут давать разные результаты. Несколько запусков с разными случайными зёрнами повышают надёжность выводов.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике сходимости линия «лучшее в популяции» перестала убывать после 25-го поколения, хотя число поколений равно 60. Линия «среднее по популяции» колеблется около того же значения. Что это означает о состоянии популяции и какие два параметра следует изменить в первую очередь?
2. Вы запустили алгоритм трижды с разными `seed` и получили три разных значения минимума: 0.003, 0.41 и 1.82. Можно ли доверять лучшему из трёх результатов? Как следует докладывать результат оптимизации в научной публикации?
3. Ваша целевая функция — результат численного моделирования, каждый вызов занимает 10 минут, и бюджет составляет 500 вызовов. Оправдан ли эволюционный алгоритм с популяцией 80 особей и 60 поколениями (4800 вызовов)? Предложите стратегию адаптации под данный бюджет.

<details>
<summary>Показать ориентиры для ответов</summary>

1. Популяция исчерпала разнообразие: все особи сконцентрировались в одной области пространства поиска, и дальнейшая эволюция не приносит улучшений. Первые шаги: (а) увеличить масштаб мутации `scale` — чтобы особи снова «прыгали» по пространству; (б) перезапустить с большей популяцией `pop_size` или с миграцией (островная модель), чтобы поддерживать разнообразие дольше.
2. Да, лучшему значению 0.003 можно доверять как найденной оценке оптимума, но нельзя считать его гарантированным глобальным минимумом. В публикации следует указывать: лучшее найденное значение, медиану и стандартное отклонение по нескольким запускам, число запусков и общий бюджет вычислений — это даёт честную оценку надёжности метода.
3. Нет, 4800 вызовов при 10-минутном времени каждого — 800 часов. При ограниченном бюджете 500 вызовов следует: сократить популяцию до 20–30 особей и число поколений до 15–20; рассмотреть байесовскую оптимизацию (например, на основе гауссовского процесса) — она эффективнее при дорогостоящих функциях, так как строит суррогатную модель и минимизирует число вызовов; применить латинский гиперкуб или квазислучайную инициализацию для лучшего начального покрытия.
</details>
